### This script prepares input files for machine learning models:
1. merging exported nutrients and catchment attributes
2. calculating Redfield normalized nutrient concentrations (as calculated for the ternary plots)

In [22]:
# load required modules:
import pandas as pd
import numpy as np
import os

In [23]:
# define the output folder path:
output_folder = '../../output_data/input_ML_learning'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

### Define functions and variables:

In [24]:
# Calculate mol ----
c_gmol = 12.01
n_gmol = 14.01
p_gmol = 30.97


In [25]:
# functions normalization redfield ratio: 
def c_imbal_eq_rfr(cmol,nmol,pmol):
    return (cmol / 106) / ((cmol / 106) + (nmol / 16) + (pmol))

def n_imbal_eq_rfr(cmol, nmol, pmol):
    return (nmol / 16) / ((cmol / 106) + (nmol / 16) + (pmol))

def p_imbal_eq_rfr(cmol,nmol,pmol):
    return pmol / ((cmol / 106) + (nmol / 16) + (pmol))
 


### load all required input data: C:N:P export and catchment attributes:

In [26]:
# 1.) load C:N:P data and prepare as input: 

##### Read in input data:-----
exp_data_daily = pd.read_csv("../../output_data/CNP_data_catchments/basin_exp_DOC_TOC_DIN_SRP_daily_median_mgL_selected.csv")

exp_data_daily.drop(columns =["obs_date"], inplace = True)


In [27]:
exp_data_daily.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 210654 entries, 0 to 210653
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   HYBAS_ID          210654 non-null  int64  
 1   NO3N_median       209867 non-null  float64
 2   NO2N_median       187618 non-null  float64
 3   NH4N_median       210654 non-null  float64
 4   NO2N_NO3N_median  787 non-null     float64
 5   TOC_median        130326 non-null  float64
 6   DOC_median        186263 non-null  float64
 7   DOC_TOC_median    210654 non-null  float64
 8   DIN_median        210654 non-null  float64
 9   SRP_median        210654 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 16.1 MB


In [28]:

# prepare exported C:N:P data:

# add column of bioavailable OC: assumption 10.08 % are bioavailable: 

exp_data_daily["DOC_TOC_bioav"] = exp_data_daily["DOC_TOC_median"] * 0.1008
 
median_HYBAS_SRP_agg = exp_data_daily.groupby('HYBAS_ID',  as_index=False).median()

median_HYBAS_SRP_agg.rename(columns={
    'DOC_TOC_bioav': 'median_DOC_TOC_bioav',
    'DIN_median': 'median_DIN',
    'SRP_median': 'median_SRP'}, inplace=True)


# convert exported nutrients from mg/L to g/L, and subsequently to mol/L and finally molar ratios relative to the Redfield ratio: 

data_rfr_srp = median_HYBAS_SRP_agg.copy() 

# convert concentrations from mg/L to g/L
data_rfr_srp["DOC_TOC_gL_bioav"] = data_rfr_srp["median_DOC_TOC_bioav"]/1000
data_rfr_srp["DIN_gL"] = data_rfr_srp["median_DIN"]/1000
data_rfr_srp["SRP_gL"] = data_rfr_srp["median_SRP"]/1000

# convert to mol per liter:

data_rfr_srp["docb_mol"] = data_rfr_srp["DOC_TOC_gL_bioav"]/c_gmol
data_rfr_srp["nreact_mol"] = data_rfr_srp["DIN_gL"]/n_gmol
data_rfr_srp["srp_mol"] = data_rfr_srp["SRP_gL"]/p_gmol

# normalize molar  concentrations relative the redfield ratio:

data_rfr_srp["c_imbal"] = c_imbal_eq_rfr(cmol = data_rfr_srp["docb_mol"], nmol = data_rfr_srp["nreact_mol"], pmol = data_rfr_srp["srp_mol"])
data_rfr_srp["n_imbal"] = n_imbal_eq_rfr(cmol = data_rfr_srp["docb_mol"], nmol = data_rfr_srp["nreact_mol"], pmol = data_rfr_srp["srp_mol"])
data_rfr_srp["p_imbal"] = p_imbal_eq_rfr(cmol = data_rfr_srp["docb_mol"], nmol = data_rfr_srp["nreact_mol"], pmol = data_rfr_srp["srp_mol"])



###  select(HYBAS_ID, c_imbal, n_imbal, p_imbal)
data_rfr_srp = data_rfr_srp[['HYBAS_ID', 'c_imbal', 'n_imbal', 'p_imbal', 'median_DOC_TOC_bioav', 'median_DIN', 'median_SRP']]




In [29]:
data_rfr_srp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3496 entries, 0 to 3495
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   HYBAS_ID              3496 non-null   int64  
 1   c_imbal               3496 non-null   float64
 2   n_imbal               3496 non-null   float64
 3   p_imbal               3496 non-null   float64
 4   median_DOC_TOC_bioav  3496 non-null   float64
 5   median_DIN            3496 non-null   float64
 6   median_SRP            3496 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 191.3 KB


### 2.) read in catchment attributes: 

In [30]:


basin_atlas_lev12_available = pd.read_csv("../../output_data/CNP_data_catchments/basin_atlas_v10_l12_data_available.csv")


# extract latitude of basins: 
export_all = pd.read_csv("../../output_data/CNP_data_catchments/DOC_TOC_DIN_SRP_daily_median_sd_subbasins.csv")

basin_lat = export_all[['HYBAS_ID', 'centroid_lat']]
basin_lat = basin_lat.drop_duplicates(subset = 'HYBAS_ID', keep = "first")

# load TWI90 values:

twi90 = pd.read_csv("../../output_data/TWI_QGIS_and_TWI90_extraction/TWI90_DOC_TOC_DIN_SRP_subbasins.csv")
twi90.drop(columns = 'Unnamed: 0', inplace = True)  



In [31]:
# merge basin_lat, twi90 and basin_atlas_lev12_available

# use basin_lat as primary df, because this df comprises all 3496 catchments for which we have data: 

HYBAS_att = pd.merge(basin_lat, twi90, how="left", on=["HYBAS_ID"])

HYBAS_att2 = pd.merge(HYBAS_att, basin_atlas_lev12_available, on = ["HYBAS_ID"])



In [32]:
### Hydrological descriptors are quite dependent of the area--> bigger catchments will have more discharge, river geometries will be higher, etc.
### Therefore, these descriptors are divided by the area (UP_AREA, see HydroBASIN documentation for details)

In [33]:
# discharge, STREAM_VOLUME, STREAM_AREA durch UP_AREA teilen--> sann catchment area behalten
# 'dis_m3_pyr' --> mean annual discharge
# 'dis_m3_pmn' --> min annual discharge
# 'dis_m3_pmx' --> max annual discharge
# 'riv_tc_usu' --> river volume
# 'ria_ha_usu --> river area in hectares
# mean annual discharge:
HYBAS_att2['dis_m3_pyr_perkm2'] = HYBAS_att2['dis_m3_pyr']/HYBAS_att2['UP_AREA']
# min annual discharge:
HYBAS_att2['dis_m3_pmn_perkm2'] = HYBAS_att2['dis_m3_pmn']/HYBAS_att2['UP_AREA']
# max annual discharge:
HYBAS_att2['dis_m3_pmx_perkm2'] = HYBAS_att2['dis_m3_pmx']/HYBAS_att2['UP_AREA']
# river volume: 
HYBAS_att2['riv_tc_usu_perkm2'] = HYBAS_att2['riv_tc_usu']/HYBAS_att2['UP_AREA']
# river area: 
HYBAS_att2['ria_ha_usu_perkm2'] = HYBAS_att2['ria_ha_usu']/HYBAS_att2['UP_AREA']

In [34]:
HYBAS_att2[['dis_m3_pyr_perkm2', 'dis_m3_pyr', 'UP_AREA', 'HYBAS_ID']].head(10)

,dis_m3_pyr_perkm2,dis_m3_pyr,UP_AREA,HYBAS_ID
0,0.009813,202.244,20609.9,1121145450
1,0.010192,191.909,18829.3,1121146530
2,0.013567,1.491,109.9,1121150540
3,0.013003,3.447,265.1,1121151290
4,0.006062,194.205,32034.9,1121151840
5,0.010710,2.382,222.4,1121153210
6,0.009226,123.153,13348.7,1121154110
7,0.011585,5.453,470.7,1121161880
8,0.010340,75.821,7333.0,1121163420
9,0.005166,177.722,34403.8,1121164110


In [35]:
HYBAS_att2.rename(columns = {'centroid_lat':'latitude'}, inplace = True)


In [36]:
HYBAS_att2.columns

Index(['HYBAS_ID', 'latitude', 'twi90', 'Unnamed: 0', 'NEXT_DOWN', 'NEXT_SINK',
       'MAIN_BAS', 'DIST_SINK', 'DIST_MAIN', 'SUB_AREA',
       ...
       'gdp_ud_usu', 'hdi_ix_sav', 'Shape_Length', 'Shape_Area', 'geometry',
       'dis_m3_pyr_perkm2', 'dis_m3_pmn_perkm2', 'dis_m3_pmx_perkm2',
       'riv_tc_usu_perkm2', 'ria_ha_usu_perkm2'],
      dtype='object', length=305)

In [37]:


#### now create a selection of attributes of HYBAS_att2

# latitude --> geographical latitude
# twi90 --> topographic wetness index
# UP_AREA --> upstream area --> see details in technical documentation of HydroBASINS
# 'slp_dg_uav' --> slope
# 'for_pc_use' --> forest extent
# 'crp_pc_use' --> cropland extent
# 'pst_pc_use' --> pasture extent
# 'urb_pc_use' --> urban extent
# 'ppd_pk_uav'] --> population density
# 'dis_m3_pyr_perkm2' --> mean annual discharge per km2 of UP_AREA
# 'dis_m3_pmn_perkm2' --> min annual discharge per km2 of UP_AREA
# 'dis_m3_pmx_perkm2' --> max annual discharge per km2 of UP_AREA
# 'run_mm_syr' --> land surface runoff
# 'inu_pc_umn' --> Inundation extent 
# 'inu_pc_umx' --> inundation extent annual max
# 'inu_pc_ult' -->Inundationn extent long time max
# 'dor_pc_pva' --> degree of regulation
# 'ria_ha_usu_perkm2 --> river area in hectares per km2 of UP_AREA
# 'riv_tc_usu_perkm2' --> river volume per km2 of UP_AREA
# 'gwt_cm_sav' --> groundwater table depth
# 'ele_mt_uav' --> elevation average meters
# 'sgr_dk_sav' --> stream gradient
# 'tmp_dc_uyr' --> air temperature
# 'pre_mm_uyr' -->precipitation
# 'pet_mm_uyr' --> potential evapotranspiration
# 'aet_mm_uyr' --> actual evapotranspiration 
# 'ari_ix_uav' --> global aridity index
# 'cmi_ix_uyr' climate_moisture_index
# 'snw_pc_uyr' --> snow cover extent
# 'wet_pc_ug2' --> wetland extent without lakes and rivers
# 'ire_pc_use' --> irrigated area extent
# gla_pc_use --> glacier extent
# prm_pc_use --> permafrost extent
# pac_pc_use --> protected area extent
# 'cly_pc_uav' --> clay fraction in soil 
# 'slt_pc_uav' --> silt fraction in soil 
# 'snd_pc_uav' --> sand fraction in soil 
# 'soc_th_uav' --> organic carbon content in soil 
# 'swc_pc_uyr' --> soil water content
# 'kar_pc_use' --> karst area extent
# 'ero_kh_uav' --> soil erosion 
### 'pop_ct_usu' --> population count --> excluded because this descriptor is dependent of catchment area, population density is used instead
# 'hft_ix_u09' -->human footprint in 2009 also u93 of 1993 available
# 'gdp_ud_usu' --> gross domnestic product in US dollars
# 'hdi_ix_sav' --> human developement index

#list_features = ['HYBAS_ID','latitude','UP_AREA' ,'twi90', 'slp_dg_uav', 'for_pc_use', 'crp_pc_use', 'pst_pc_use', 'urb_pc_use', 'ppd_pk_uav',
#                'dis_m3_pyr_perkm2', 'dis_m3_pmn_perkm2', 'dis_m3_pmx_perkm2', 'run_mm_syr', 'inu_pc_umn', 'inu_pc_umx', 'inu_pc_ult', 'dor_pc_pva',
#                'ria_ha_usu_perkm2', 'riv_tc_usu_perkm2',
#                'gwt_cm_sav', 'ele_mt_uav', 'sgr_dk_sav', 'tmp_dc_uyr', 'pre_mm_uyr', 'pet_mm_uyr', 'aet_mm_uyr', 'ari_ix_uav',
#                'cmi_ix_uyr', 'snw_pc_uyr', 'wet_pc_ug2', 'ire_pc_use', 'gla_pc_use', 'prm_pc_use', 'pac_pc_use', 'cly_pc_uav',
#                'slt_pc_uav', 'snd_pc_uav', 'soc_th_uav', 'swc_pc_uyr', 'kar_pc_use', 'ero_kh_uav', 'hft_ix_u09',
#                'gdp_ud_usu', 'hdi_ix_sav']



# based on meeting with Pia, Felipe and Daniel 15.05.2024: we decided to preselect feature variables: 
# based on spearman correlation matrix (see script matrix_correlation_features) several features were excluded, because they show strong correlations with other 
# variables. 
list_features_selected = ['HYBAS_ID','UP_AREA' ,'twi90', 'slp_dg_uav', 'for_pc_use', 'crp_pc_use', 'pst_pc_use', 'ppd_pk_uav',
                'run_mm_syr', 'inu_pc_umn', 'dor_pc_pva',
                'ria_ha_usu_perkm2', 'riv_tc_usu_perkm2',
                'ele_mt_uav', 'sgr_dk_sav', 'tmp_dc_uyr', 'pre_mm_uyr', 'pet_mm_uyr', 'aet_mm_uyr',
                'snw_pc_uyr', 'wet_pc_ug2', 'ire_pc_use', 'gla_pc_use', 'prm_pc_use', 'pac_pc_use', 'cly_pc_uav',
                'slt_pc_uav', 'snd_pc_uav', 'soc_th_uav', 'swc_pc_uyr', 'kar_pc_use', 'ero_kh_uav',
                'gdp_ud_usu', 'hdi_ix_sav']



# filter only selected features
basin_features = HYBAS_att2[list_features_selected]




# response variables are contained in data_rfr_srp
# predictive variables are contained in basin_features


# MERGE tables to obtain one large dataframe containing response values and predictive variables

data_export_basins = pd.merge(data_rfr_srp, basin_features, on = ['HYBAS_ID'])





In [38]:
print(f"After preselection, {len(list_features_selected)} features remain.")

After preselection, 34 features remain.


In [39]:
data_export_basins

,HYBAS_ID,c_imbal,n_imbal,p_imbal,median_DOC_TOC_bioav,median_DIN,median_SRP,UP_AREA,twi90,slp_dg_uav,...,pac_pc_use,cly_pc_uav,slt_pc_uav,snd_pc_uav,soc_th_uav,swc_pc_uyr,kar_pc_use,ero_kh_uav,gdp_ud_usu,hdi_ix_sav
0,1121145450,0.029363,0.501158,0.469480,0.084407,0.253667,0.032831,20609.9,8.826505,48,...,20,32,23,44,27,49,0,11117,1.175299e+10,555
1,1121146530,0.007609,0.583281,0.409110,0.100636,1.358399,0.131635,18829.3,8.074445,50,...,18,32,23,44,28,51,0,11470,1.130367e+10,555
2,1121150540,0.031992,0.735331,0.232677,0.054268,0.219630,0.009602,109.9,8.204151,67,...,22,36,26,39,35,68,0,12078,6.879562e+07,555
3,1121151290,0.013639,0.113814,0.872547,0.072643,0.106733,0.113051,265.1,9.421020,89,...,59,33,26,41,56,75,0,11736,8.293591e+07,555
4,1121151840,0.025477,0.314462,0.660061,0.114009,0.247784,0.071857,32034.9,9.312500,35,...,27,32,24,44,22,38,0,7784,1.263985e+10,555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3491,8120253120,0.096346,0.379972,0.523682,0.302400,0.209996,0.039986,80185.4,6.915997,91,...,10,13,42,43,106,85,2,1660,4.468250e+07,948
3492,8120256970,0.045640,0.882384,0.071976,0.052477,0.178642,0.002013,111.5,5.486714,222,...,62,7,34,58,148,89,0,3379,2.679400e+05,948
3493,8120280400,0.068245,0.797126,0.134629,0.072929,0.149992,0.003500,5195.2,15.634457,128,...,88,10,37,52,122,91,0,2878,9.341507e+08,948
3494,8120295370,0.380209,0.272594,0.347197,0.720821,0.090998,0.016013,180.3,8.519233,33,...,0,10,45,45,133,89,0,64,6.378524e+07,948


# Save data:

In [40]:


data_export_basins.to_csv("../../output_data/input_ML_learning/median_cnp_export_abs_conc_and_rfr_and_basin_feat.csv")


HYBAS_att2.to_csv("../../output_data/input_ML_learning/selected_basin_features.csv")

data_rfr_srp.to_csv("../../output_data/input_ML_learning/median_cnp_export_abs_conc_and_rfr_normalized.csv")


